# Cardiac Disease Exploratory Data Analysis
This notebook compares three cardiac datasets used in our fairness research pipeline: Cleveland, Kaggle Heart, and Cardio70k.

**Purpose:** assess data quality, representation, and distribution consistency before baseline modeling and fairness evaluation.

**Key questions:**
1. Who is represented across datasets (age, sex, and intersections)?
2. Which features and targets shift between datasets?
3. Where can data issues bias downstream models (missingness and outliers)?

## Imports
Load required libraries for data handling, plotting, and lightweight statistics.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT_DIR = Path.cwd().resolve()
if (ROOT_DIR / "configs").exists() is False:
    ROOT_DIR = ROOT_DIR.parents[0]
sys.path.append(str(ROOT_DIR / "src"))

from fairxai.notebook_utils import (
    set_schema_cfg,
    detect_csv_sep,
    dataset_age_unit,
    age_group_order,
    apply_age_group_order,
    age_to_years,
    resolve_sex_series,
    load_domain_config,
    get_relevant_datasets,
    make_figure_path_builder,
    load_external_datasets,
    load_raw_datasets,
    load_processed_scaled_datasets,
    summarize_stage,
    canonical_features_for_columns,
 )
from fairxai.viz import (
    PALETTE_DATASET,
    PALETTE_SEX,
    PALETTE_TARGET,
    UNITS,
    plot_categorical_distribution_grid,
    plot_numeric_distribution_comparison,
    plot_target_distribution_by_group,
    plot_stacked_group_distribution_grid,
    plot_correlation_heatmap_grid,
    plot_pca_kmeans_scatter_grid,
    plot_two_dataset_feature_distributions,
    summarize_ks_test_between_datasets,
    plot_drift_heatmap,
    plot_missing_data_patterns,
    plot_outlier_analysis,
    plot_mixed_feature_batches,
    plot_bmi_and_bp_relationship,
    CARDIAC_CATEGORY_VALUE_LABEL_MAPPING,
    CARDIAC_CATEGORY_DISPLAY_ORDER,
    normalize_cardiac_category_series,
 )

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid", context="notebook")
%load_ext mermaid_magic

## Paths and configuration
Define project paths, load pipeline config, and set shared color mappings.

In [ ]:
MEDICAL_AREA = "cardiac"
NOTEBOOK_TYPE = "eda"
SAVE_FIGURES = True

cfg = load_domain_config(ROOT_DIR, MEDICAL_AREA)
feature_map = cfg["feature_map"]
schema_cfg = cfg["schema_cfg"]

DATA_DIR = ROOT_DIR / "data" / MEDICAL_AREA
RESULTS_DIR = ROOT_DIR / "output" / MEDICAL_AREA
EXTERNAL_DIR = cfg["external_dir"]
RAW_DIR = cfg["raw_dir"]
PROCESSED_DIR = cfg["processed_dir"]

DATASETS = get_relevant_datasets(schema_cfg, MEDICAL_AREA)
PRIMARY_DATASETS = ["cleveland", "kaggle_heart"]
TARGET_COL = feature_map["target"]["heart_disease"]["canonical"]

FIGURES_DIR, fig_path = make_figure_path_builder(ROOT_DIR, MEDICAL_AREA, NOTEBOOK_TYPE)

set_schema_cfg(schema_cfg)

ROOT_DIR, DATA_DIR, RESULTS_DIR, FIGURES_DIR

## Data pipeline overview
Track how each dataset moves from downloaded files to canonical raw data and then to model-ready splits.

Raw downloads are standardized into a canonical schema, then preprocessed into train/test splits with scaling and metadata artifacts.

In [ ]:
%%mermaid
flowchart TD
    A[External files] --> B[Raw canonical schema]
    B --> C[Preprocessed train/test splits]
    B --> D[EDA diagnostics]
    D --> E[Representation findings]
    D --> F[Data quality findings]

# Section A — Data staging, missingness, and outliers
This section validates dataset alignment and surfaces early quality risks before representation and target analyses.

## Load external, raw, and processed datasets
Read each dataset in its lifecycle stages to enable before/after comparisons.

In [ ]:
EXTERNAL_FILES = {
    "cleveland": EXTERNAL_DIR / "heart_cleveland_upload.csv",
    "kaggle_heart": EXTERNAL_DIR / "heart.csv",
    "cardio70k": EXTERNAL_DIR / "cardio_train.csv",
}

external = load_external_datasets(EXTERNAL_FILES, detect_csv_sep=detect_csv_sep)
raw = load_raw_datasets(RAW_DIR, DATASETS)
processed = load_processed_scaled_datasets(PROCESSED_DIR, DATASETS)

print("\nDataset shapes across formats:")
for name in DATASETS:
    ext_shape = external[name].shape if name in external else "N/A"
    raw_shape = raw[name].shape if name in raw else "N/A"
    proc_train_shape = processed[name]["train"].shape if name in processed else "N/A"
    proc_test_shape = processed[name]["test"].shape if name in processed else "N/A"
    print(f"{name}: External={ext_shape}, Raw={raw_shape}, Processed Train={proc_train_shape}, Processed Test={proc_test_shape}")

## Outlier detection across datasets
Replace dataset-specific sanity checks with a unified outlier screen across Cleveland, Kaggle, and Cardio70k.

In [ ]:
outlier_datasets = {}
for name, df in raw.items():
    df_copy = df.copy()
    if "age_raw" in df_copy.columns:
        unit = dataset_age_unit(name)
        df_copy["age_raw"] = age_to_years(df_copy["age_raw"], unit)
    outlier_datasets[name] = df_copy

candidate_features = [
    "age_raw",
    "trestbps",
    "chol",
    "thalach",
    "ap_hi",
    "ap_lo",
    "height",
    "weight",
]
outlier_features = [
    feature
    for feature in candidate_features
    if any(feature in frame.columns for frame in outlier_datasets.values())
]

if outlier_features:
    _ = plot_outlier_analysis(
        datasets=outlier_datasets,
        features=outlier_features,
        method="clinical",
        save_path=fig_path("outlier_analysis") if SAVE_FIGURES else None,
        show=False,
    )
else:
    print("No supported numeric features found for outlier analysis.")

💡 **What to look for:** values outside red reference lines are clinically implausible and should be reviewed as possible data-entry issues, coding artifacts, or rare-but-valid edge cases.

## Data quality and cleaning impact
Compare external vs raw to quantify row drops and missingness.

In [ ]:
summary_external = summarize_stage(external, "external")
summary_raw = summarize_stage(raw, "raw")
cleaning_summary = summary_external.merge(summary_raw, on="dataset", suffixes=("_external", "_raw"))
cleaning_summary["rows_removed"] = cleaning_summary["rows_external"] - cleaning_summary["rows_raw"]
cleaning_summary["pct_removed"] = (cleaning_summary["rows_removed"] / cleaning_summary["rows_external"]).round(4)
cleaning_summary[["dataset", "rows_external", "rows_raw", "rows_removed", "pct_removed", "rows_with_missing_external"]]

**Note:** The external Cleveland file used here (297 rows) has already been cleaned.

The original UCI Cleveland dataset had 303 rows; 6 were removed due to missing values in 'ca' and 'thal' columns during initial data preparation.

## Missing data pattern analysis
Visualize missingness rates and co-occurrence structure across all raw datasets before feature-level comparisons.

In [ ]:
_ = plot_missing_data_patterns(
    datasets={
        "cleveland": raw["cleveland"],
        "kaggle_heart": raw["kaggle_heart"],
        "cardio70k": raw["cardio70k"],
    },
    title="Missing data patterns across datasets",
    save_path=fig_path("missing_data_patterns") if SAVE_FIGURES else None,
    show=False,
)

💡 **Why this matters:** non-random missingness can bias subgroup performance, and co-missing features often reveal collection-process issues that should be addressed before modeling.

## Feature schema comparison (canonical mapping)
Use the YAML feature map to compare canonical features rather than raw column names.

In [ ]:
canonical_by_dataset = {}
for name, df in raw.items():
    canonical_by_dataset[name] = sorted(canonical_features_for_columns(list(df.columns), name, feature_map=feature_map))
    print(f"\n{name} canonical features ({len(canonical_by_dataset[name])}):")
    print(canonical_by_dataset[name])

shared_all = set.intersection(*[set(v) for v in canonical_by_dataset.values()])
primary_sets = [set(canonical_by_dataset[name]) for name in PRIMARY_DATASETS if name in canonical_by_dataset]
shared_primary = set.intersection(*primary_sets) if primary_sets else set()
unique_canonical = {
    name: sorted(set(feats) - shared_all) for name, feats in canonical_by_dataset.items()
}

print(f"\nShared canonical features across all datasets ({len(shared_all)}):")
print(sorted(shared_all))
print(f"Shared canonical features across primary datasets only ({len(shared_primary)}):")
print(sorted(shared_primary))
for name, feats in unique_canonical.items():
    print(f"{name} unique canonical features ({len(feats)}): {feats}")

**Note:** Cleveland is a subset inside the Kaggle Heart compilation; Cardio70k follows a different schema.

# Section B — Representation & Sensitive Attributes
The focus here is who is represented in each dataset (age and sex) and where subgroup imbalance may impact downstream model behavior.

## Sensitive attribute analysis: age
Compare age distributions with unit awareness across datasets.

In [ ]:
age_units = []
for name, df in raw.items():
    unit = dataset_age_unit(name)
    age_units.append({"dataset": name, "age_unit": unit})
pd.DataFrame(age_units)

In [ ]:
def age_years_getter(name: str, df: pd.DataFrame) -> pd.Series:
    unit = dataset_age_unit(name)
    return age_to_years(df["age_raw"], unit)

age_datasets = {name: raw[name] for name in DATASETS if "age_raw" in raw[name].columns}
fig, axes = plot_numeric_distribution_comparison(
    datasets=age_datasets,
    column="age_raw",
    title="Age distribution across datasets (years)",
    bins=20,
    colors=PALETTE_DATASET,
    value_getter=age_years_getter,
    xlabel="age (years)",
    save_path=fig_path("age_distribution_years") if SAVE_FIGURES else None,
    show=False,
 )

for idx, name in enumerate(age_datasets):
    unit = dataset_age_unit(name)
    suffix = " (converted from days)" if unit == "days" else ""
    axes[idx].set_title(f"{name} age (years){suffix}")

### Age groups (raw standardized)
Compare standardized age_group distributions for each dataset.

In [ ]:
age_group_datasets = {}
combined_age_order = []
for name in DATASETS:
    df = raw[name]
    if "age_group" not in df.columns:
        continue
    order = age_group_order(name)
    for label in order:
        if label not in combined_age_order:
            combined_age_order.append(label)
    age_group_datasets[name] = pd.DataFrame({
        "age_group": apply_age_group_order(df["age_group"], name).astype(str),
    })

_ = plot_categorical_distribution_grid(
    datasets=age_group_datasets,
    column="age_group",
    title="Age-group counts across datasets",
    subtitle="Age-group representation changes by source cohort and affects subgroup reliability.",
    palette=PALETTE_DATASET,
    category_order=combined_age_order,
    show_percentages=True,
    annotate_imbalance=False,
    save_path=fig_path("age_group_counts") if SAVE_FIGURES else None,
    show=False,
 )

💡 **Why this matters:** Uneven age-group representation can make model behavior look strong overall while hiding weaker performance in underrepresented age bands.

In [ ]:
age_stats = []
age_plot_rows = []
for name, df in raw.items():
    unit = dataset_age_unit(name)
    age_years = age_to_years(df["age_raw"], unit)
    age_stats.append({
        "dataset": name,
        "age_unit": unit,
        "min": age_years.min(),
        "max": age_years.max(),
        "mean": age_years.mean(),
        "median": age_years.median(),
        "std": age_years.std(),
    })
    age_plot_rows.append(pd.DataFrame({"dataset": name, "age_years": age_years}))

age_stats_df = pd.DataFrame(age_stats)
age_plot = pd.concat(age_plot_rows, ignore_index=True)
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=age_plot, x="dataset", y="age_years", ax=ax, palette=PALETTE_DATASET)
ax.set_title("Age distribution comparison (years)", fontsize=12, fontweight="bold")
ax.set_xlabel("Dataset")
ax.set_ylabel("Age (years)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(fig_path("age_boxplot"), dpi=300, bbox_inches="tight")

display(age_stats_df)

## Sensitive attribute analysis: sex
Compare sex encoding and distribution across datasets.

In [ ]:
sex_tables = []
for name, df in raw.items():
    series = resolve_sex_series(df)
    if series is None:
        continue
    counts = series.value_counts(dropna=False)
    total = counts.sum()
    for label, value in counts.items():
        sex_tables.append({
            "dataset": name,
            "sex": label,
            "count": int(value),
            "pct": round(value / total, 4),
        })
sex_df = pd.DataFrame(sex_tables)
display(sex_df)

In [ ]:
sex_plot_datasets = {}
for name in DATASETS:
    series = resolve_sex_series(raw[name])
    if series is not None:
        sex_plot_datasets[name] = pd.DataFrame({"sex": series.astype(str)})

_ = plot_categorical_distribution_grid(
    datasets=sex_plot_datasets,
    column="sex",
    title="Sex counts across datasets",
    palette=PALETTE_SEX,
    show_percentages=True,
    annotate_imbalance=False,
    save_path=fig_path("sex_counts") if SAVE_FIGURES else None,
    show=False,
 )

## Intersectional view: age group x sex
Compare age-group composition by sex within each dataset.

In [ ]:
intersectional_datasets = {}
age_order_by_dataset = {}
for name in DATASETS:
    df = raw[name]
    series = resolve_sex_series(df)
    if "age_group" not in df.columns or series is None:
        continue
    intersectional_datasets[name] = pd.DataFrame({
        "age_group": apply_age_group_order(df["age_group"], name).astype(str),
        "sex": series.astype(str),
    })
    age_order_by_dataset[name] = [str(value) for value in age_group_order(name)]

_ = plot_stacked_group_distribution_grid(
    datasets=intersectional_datasets,
    group_col="age_group",
    stack_col="sex",
    title="Age-group composition by sex",
    subtitle="Intersectional composition helps reveal underrepresented subgroup pockets.",
    group_order_by_dataset=age_order_by_dataset,
    stack_order=["Female", "Male"],
    stack_palette=PALETTE_SEX,
    annotate_totals=True,
    save_path=fig_path("age_group_by_sex") if SAVE_FIGURES else None,
    show=False,
 )

# Section C — Target Prevalence Patterns
These plots isolate baseline prevalence differences by subgroup, which should be interpreted separately from model fairness metrics.

## Target variable overview
Show heart disease prevalence overall and by sensitive attributes.

In [ ]:
target_rows = []
for name, df in raw.items():
    if TARGET_COL in df.columns:
        target_rows.append({"dataset": name, "prevalence": float(df[TARGET_COL].mean())})
pd.DataFrame(target_rows)

In [ ]:
prevalence_by_sex = {}
for name in DATASETS:
    df = raw[name]
    series = resolve_sex_series(df)
    if TARGET_COL in df.columns and series is not None:
        prevalence_by_sex[name] = pd.DataFrame({"sex": series.astype(str), TARGET_COL: df[TARGET_COL]})

_ = plot_target_distribution_by_group(
    datasets=prevalence_by_sex,
    target_col=TARGET_COL,
    group_col="sex",
    title="Heart disease prevalence by sex",
    palette=PALETTE_SEX,
    kind="bar",
    save_path=fig_path("prevalence_by_sex") if SAVE_FIGURES else None,
    show=False,
 )

In [ ]:
prevalence_by_age_group = {}
for name in DATASETS:
    df = raw[name]
    if TARGET_COL not in df.columns or "age_group" not in df.columns:
        continue
    prevalence_by_age_group[name] = pd.DataFrame({
        "age_group": apply_age_group_order(df["age_group"], name),
        TARGET_COL: df[TARGET_COL],
    })

_ = plot_target_distribution_by_group(
    datasets=prevalence_by_age_group,
    target_col=TARGET_COL,
    group_col="age_group",
    title="Heart disease prevalence by age group",
    kind="line",
    save_path=fig_path("prevalence_by_age_group") if SAVE_FIGURES else None,
    show=False,
 )

💡 **Why this matters:** Prevalence trends by age group show baseline risk differences that should be separated from true model fairness effects.

## Clinical feature distributions (Cleveland vs Kaggle)
Compare shared clinical features for the two primary datasets.

In [ ]:
shared_common = [info["canonical"] for info in feature_map.get("common", {}).values()]
cleveland_df = raw["cleveland"]
kaggle_df = raw["kaggle_heart"]

plot_features = [feature for feature in shared_common if feature in cleveland_df.columns and feature in kaggle_df.columns]

distribution_summary = plot_two_dataset_feature_distributions(
    dataset_a=cleveland_df,
    dataset_b=kaggle_df,
    shared_features=plot_features,
    dataset_a_name="cleveland",
    dataset_b_name="kaggle_heart",
    dataset_palette=PALETTE_DATASET,
    units=UNITS,
    categorical_feature_names=set(CARDIAC_CATEGORY_VALUE_LABEL_MAPPING.keys()),
    categorical_series_normalizer=normalize_cardiac_category_series,
    categorical_order_map=CARDIAC_CATEGORY_DISPLAY_ORDER,
    save_path=fig_path("clinical_distribution") if SAVE_FIGURES else None,
    show=False,
 )

numeric_features = distribution_summary["numeric_features"]
categorical_features = distribution_summary["categorical_features"]

In [ ]:
print("\n🔬 Distribution Drift Analysis (Cleveland vs Kaggle)")
ks_results = summarize_ks_test_between_datasets(
    dataset_a=cleveland_df,
    dataset_b=kaggle_df,
    features=numeric_features,
    dataset_a_name="cleveland",
    dataset_b_name="kaggle_heart",
    alpha=0.05,
    min_unique_values=2,
)
display(ks_results)

_ = plot_drift_heatmap(
    reference_dataset=cleveland_df,
    comparison_datasets={"kaggle_heart": kaggle_df},
    features=numeric_features,
    reference_name="cleveland",
    threshold=0.05,
    save_path=fig_path("drift_heatmap_cleveland_kaggle") if SAVE_FIGURES else None,
    show=False,
)

## Correlation structure (Cleveland vs Kaggle)
Review correlation patterns for shared clinical features.

In [ ]:
corr_targets = [("cleveland", cleveland_df, plot_features), ("kaggle_heart", kaggle_df, plot_features)]
cardio_df = raw.get("cardio70k")
if cardio_df is not None:
    cardio_features = [c for c in cardio_df.columns if c != TARGET_COL]
    corr_targets.append(("cardio70k", cardio_df, cardio_features))
else:
    corr_targets.append(("cardio70k", pd.DataFrame(), []))

_ = plot_correlation_heatmap_grid(
    corr_targets=corr_targets,
    categorical_feature_names=set(CARDIAC_CATEGORY_VALUE_LABEL_MAPPING.keys()),
    categorical_series_normalizer=normalize_cardiac_category_series,
    figsize=(24, 6),
    annot=True,
    annot_size=8,
    save_path=fig_path("correlation_heatmap_grid") if SAVE_FIGURES else None,
    show=False,
 )

## Clustering and similarity view (raw)
Explore potential latent groups beyond age/sex using a simple PCA + KMeans view.

In [ ]:
clustering_datasets = {name: raw[name] for name in DATASETS}
_ = plot_pca_kmeans_scatter_grid(
    datasets=clustering_datasets,
    target_col=TARGET_COL,
    n_clusters=3,
    sample_size=1500,
    random_state=42,
    figsize=(15, 4),
    save_path=fig_path("pca_kmeans_clusters") if SAVE_FIGURES else None,
    show=False,
 )

## Cardio70k-specific features
Focus on features unique to the 70k dataset and their distributions.

In [ ]:
cardio_df = raw.get("cardio70k")
if cardio_df is not None:
    cardio_unique = list(feature_map.get("dataset_specific", {}).get("cardio70k", {}).keys())
    cardio_unique = [feature for feature in cardio_unique if feature in cardio_df.columns]
    if cardio_unique:
        _ = plot_mixed_feature_batches(
            df=cardio_df,
            features=cardio_unique,
            dataset_name="cardio70k",
            color=PALETTE_DATASET["cardio70k"],
            units=UNITS,
            batch_size=4,
            categorical_unique_threshold=5,
            save_path=fig_path("cardio70k_unique_features") if SAVE_FIGURES else None,
            show=False,
        )
    else:
        print("No cardio70k-specific features found in raw data.")

In [ ]:
if cardio_df is not None:
    _ = plot_bmi_and_bp_relationship(
        df=cardio_df,
        color=PALETTE_DATASET["cardio70k"],
        height_col="height",
        weight_col="weight",
        systolic_col="ap_hi",
        diastolic_col="ap_lo",
        save_path=fig_path("cardio70k_bmi_bp") if SAVE_FIGURES else None,
        show=False,
    )

# Key findings
Interpretation of the main diagnostics and how they inform the next modeling phase.

### Representation imbalances
Clinical datasets remain male-skewed, while Cardio70k is closer to balanced. This implies subgroup reliability differs by source and should be tracked explicitly in evaluation.

### Feature drift between Cleveland and Kaggle
KS diagnostics plus drift heatmap show non-trivial distribution shift in multiple numeric clinical features, so cross-dataset generalization should be treated as a domain-shift problem.

### Data quality risk areas
Missingness and outlier plots reveal where preprocessing assumptions may fail and where clinically implausible values can distort model behavior if left untreated.

### Cardio70k feature-space mismatch
Cardio70k includes lifestyle/exam variables not present in Cleveland/Kaggle, which limits one-model-fits-all strategies and motivates dataset-specific feature handling.